In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Utility Functions and Classes


This section contains the implementations of utility functions and classes used in this book.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import collections
import inspect
from IPython import display

## Hyperparameter Utilities

In [2]:
@d2l.add_to_class(d2l.HyperParameters)
def save_hyperparameters(self, ignore=None):
    """Save function arguments into instance attributes."""
    ignore = [] if ignore is None else ignore
    frame = inspect.currentframe().f_back
    _, _, _, local_vars = inspect.getargvalues(frame)
    self.hparams = {k:v for k, v in local_vars.items()
                    if k not in set(ignore+['self']) and not k.startswith('_')}
    for k, v in self.hparams.items():
        setattr(self, k, v)

## Progress and Plotting

The `ProgressBoard` of that section draws on the calling thread.
`show` renders all curves into a new figure and replaces the previous frame
of the same board in the notebook output. In the headless book build only the
final call renders, so an executed notebook records one figure per board.
Losses go on the left axis and accuracies on a second axis on the right, in
which case the legend moves above the plot so that it covers neither set of
curves. Validation points, and any curve that has a single point, are marked
with dots, so that a one-epoch run is still visible.

In [3]:

import os
from matplotlib.figure import Figure

@d2l.add_to_class(d2l.ProgressBoard)
def show(self, final=True):
    """Redraw all curves; the book build renders only the final call."""
    if not self.display or (
            not final and os.environ.get('D2L_NB_CAPTURE') == '1'):
        return
    d2l.use_svg_display()
    fig = Figure(figsize=self.figsize)
    self._plot(fig.add_subplot())
    if self.handle is None:
        self.handle = display.display(fig, display_id=True)
    else:
        self.handle.update(fig)

@d2l.add_to_class(d2l.ProgressBoard)
def _plot(self, ax):
    """Losses on the left axis, accuracies on a right one."""
    acc_ax = None
    for i, (label, (xs, ys)) in enumerate(self.data.items()):
        target = ax
        if label.endswith('acc'):
            acc_ax = acc_ax or ax.twinx()
            target = acc_ax
        # Mark validation points, and any curve that has a single point
        marked = label.startswith('val') or len(xs) == 1
        target.plot(xs, ys, ('-', '--', '-.', ':')[i % 4], color=f'C{i % 10}',
                    marker='o' if marked else None, markersize=3,
                    clip_on=not marked, label=label)
    ax.set_xlabel(self.xlabel), ax.set_ylabel(self.ylabel)
    ax.set_xscale(self.xscale), ax.set_yscale(self.yscale)
    if self.xlim: ax.set_xlim(self.xlim)
    if self.ylim: ax.set_ylim(self.ylim)
    if self.yscale == 'log': d2l.set_log_ticks(ax.yaxis)
    if acc_ax is not None: acc_ax.set_ylabel('accuracy')
    lines = list(ax.get_lines()) + list(acc_ax.get_lines() if acc_ax else [])
    labels = [line.get_label() for line in lines]
    if acc_ax is not None:  # Above the axes: no curve of either axis is hidden
        ax.legend(lines, labels, loc='lower center', ncol=len(lines),
                  bbox_to_anchor=(0.5, 1), frameon=False)
    elif lines:
        ax.legend(lines, labels)
    ax.grid()

`show_trace_2d` draws the iterates of a two-dimensional update rule, as
`d2l.train_2d` of that section returns them, over the contour lines of
the objective `f`. The window covers the start $(-5, -2)$ and the minimum
at the origin, and widens to fit the trace. Both axes share one scale
unless the trace has diverged along one of them.

In [ ]:
def show_trace_2d(f, results):
    """Show the trace of 2D variables over the contours of f."""
    x1s, x2s = zip(*[(float(a), float(b)) for a, b in results])
    grids = []  # A window around start and minimum, widened to fit the trace
    for (lo, hi), xs in zip([(-5.5, 1.0), (-3.0, 3.0)], (x1s, x2s)):
        lo, hi = min(lo, min(xs) - 0.5), max(hi, max(xs) + 0.5)
        grids.append(d2l.arange(lo, hi, (hi - lo) / 100))
    x1, x2 = d2l.meshgrid(*grids, indexing='ij')
    width, height = (float(g[-1] - g[0]) for g in grids)
    equal = 0.25 < height / width < 4  # Equal scales unless the trace diverged
    d2l.set_figsize((3, 2.8) if equal else (3.5, 2.5))
    d2l.plt.contour(x1, x2, f(x1, x2), colors='0.6', linewidths=0.8)
    d2l.plt.plot(x1s, x2s, '-o', color='C1', markersize=4)
    d2l.plt.xlabel('$x_1$')
    d2l.plt.ylabel('$x_2$')
    if equal:
        d2l.plt.gca().set_aspect('equal')

## Reinforcement Learning Utilities

In [ ]:

def plot_curves(curves, xlabel, ylabel, smooth=1, reference=None,
                ylim=None, x=None, xscale='linear', yscale='linear',
                band='range'):
    """One panel per call. `curves` maps a label to an array of shape
    [seed, step] (or [step] for a single seed); plots the seed median and a
    shaded band over the seeds: min to max (`band='range'`) or the
    interquartile range (`band='iqr'`, the better summary from five seeds
    on). `smooth`, if greater than 1, replaces each point by the mean of the
    `smooth` steps that end there, before the seed statistics; the first
    `smooth - 1` steps lack a full window and are dropped. `x` gives the
    position of each step (default 0, 1, 2, ...). `reference`, if given,
    draws a dashed horizontal line; `ylim` fixes the y-axis range; `xscale`
    and `yscale` take matplotlib's scale names, e.g. 'log'."""
    set_figsize((6, 4))
    for name, values in curves.items():
        values = np.atleast_2d(values)
        xs = np.arange(values.shape[1]) if x is None else np.asarray(x)
        if smooth > 1:
            kernel = np.ones(smooth) / smooth
            values = np.stack([np.convolve(v, kernel, 'valid')
                               for v in values])
            xs = xs[smooth - 1:]      # each window sits at its last step
        line, = plt.plot(xs, np.median(values, axis=0), label=name)
        if values.shape[0] > 1:
            lo, hi = ((values.min(axis=0), values.max(axis=0))
                      if band == 'range' else
                      np.percentile(values, [25, 75], axis=0))
            plt.fill_between(xs, lo, hi, alpha=0.2, color=line.get_color())
    if reference is not None:
        plt.axhline(reference, linestyle='--', color='gray')
    if ylim is not None:
        plt.ylim(ylim)
    plt.xscale(xscale)
    plt.yscale(yscale)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()

def plot_hist(samples, xlabel, ylabel, bins=60, vlines=()):
    """Overlaid histograms of `samples` (label -> values), dashed at `vlines`."""
    set_figsize((6, 4))
    for name, x in samples.items():
        plt.hist(x, bins=bins, alpha=0.5, label=name)
    for v in vlines:
        plt.axvline(v, linestyle='--', color='black')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()

def show_grid(desc, values, policy=None, titles=None,
              diverging=False, fmt='{:.2f}'):
    """A gridworld in the style of the chapter's figures. Each cell is
    shaded by `values` (white to blue, or with `diverging=True` red to blue
    around zero, for differences) and shows its state index (top left), its
    value (bottom left) and an arrow for the action of `policy`, if given.
    Holes are grey and the goal green; S, H and G sit at the bottom right.
    The grid shape is read from `desc` (e.g. `env.unwrapped.desc`).
    `values` and `policy` may each be a single frame, shape (num_states,),
    or a sequence of frames, shape (num_frames, num_states); `titles`, if
    given, labels each frame."""
    h, w = desc.shape
    values = np.atleast_2d(values).reshape(-1, h * w)
    if policy is not None:
        policy = np.atleast_2d(policy).reshape(-1, h * w)
    if diverging:
        top = max(np.abs(values).max(), 1e-12)
        cmap, norm = plt.get_cmap('RdBu'), plt.Normalize(-top, top)
    else:
        cmap = plt.get_cmap('Blues')
        norm = plt.Normalize(min(0.0, values.min()),
                             max(values.max(), 1e-12))
    move = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}   # (dx, dy)
    fig, axes = plt.subplots(1, len(values), squeeze=False,
                             figsize=(3.6 * len(values), 3.6))
    for k, ax in enumerate(axes[0]):
        for s in range(h * w):
            y, x = divmod(s, w)
            cell, v = desc[y, x].decode(), values[k, s]
            fill = {'H': ('#565D66', 0.55), 'G': ('#2CA02C', 0.35)}.get(
                cell, (cmap(norm(v)), 1.0))
            ax.add_patch(plt.Rectangle((x, y), 1, 1, fc=fill[0],
                                       alpha=fill[1], ec='black', lw=1.0))
            shade = np.abs(norm(v) - 0.5) * 2 if diverging else norm(v)
            ink = 'white' if cell == 'H' or (cell != 'G' and shade > 0.6) \
                else 'black'
            ax.text(x + 0.07, y + 0.07, str(s), ha='left', va='top',
                    fontsize=9, color=ink)
            if cell not in 'HG' or np.abs(v) > 1e-9:   # terminal: 0
                ax.text(x + 0.07, y + 0.93, fmt.format(v), ha='left',
                        va='bottom', fontsize=10, color=ink)
            if cell in 'SHG':
                ax.text(x + 0.82, y + 0.80, cell, ha='center', va='center',
                        fontsize=13, fontweight='bold', color=ink)
            if policy is not None and cell not in 'HG':
                dx, dy = move[int(policy[k, s])]
                ax.annotate('', xy=(x + 0.5 + 0.22 * dx, y + 0.45 + 0.22 * dy),
                            xytext=(x + 0.5 - 0.22 * dx,
                                    y + 0.45 - 0.22 * dy),
                            arrowprops=dict(arrowstyle='-|>', color=ink,
                                            lw=1.6, mutation_scale=14))
        ax.set_xlim(0, w)
        ax.set_ylim(h, 0)
        ax.set_aspect('equal')
        ax.axis('off')
        if titles is not None:
            ax.set_title(titles[k])
    fig.tight_layout()

def run_seeds(train, num_seeds, **kwargs):
    """Run train(seed, **kwargs), a generator of curve points, per seed."""
    return np.array([list(train(seed, **kwargs)) for seed in range(num_seeds)])

def on_grid(runs, grid, col, smooth=1):
    """Resample per-episode rows (env step, value, ...) onto an env-step
    grid. For each run, drop the rows whose column `col` is NaN, average
    the column over the last `smooth` episodes, and interpolate linearly at
    the grid points; returns [run, grid point], the input of plot_curves."""
    out = []
    for r in runs:
        r = np.asarray(r, dtype=float)
        r = r[~np.isnan(r[:, col])]
        v = np.convolve(r[:, col], np.ones(smooth) / smooth, 'valid')
        out.append(np.interp(grid, r[smooth - 1:, 0], v))
    return np.stack(out)

`run_seeds` runs a training generator once per seed and stacks the curves it yields into one array of shape (seed, step, ...), the input `plot_curves` expects. Off-policy agents report once per episode, and seeds finish different numbers of episodes; `on_grid` puts such per-episode records on a common grid of environment steps, so that they too can be stacked and plotted.

The JAX implementations of the reinforcement-learning agents act one state at a time and update on batches whose length changes from one batch to the next. Compiled code is keyed on the structure of the network, the shapes of its inputs and the optimizer, so three helpers keep the number of compilations small. `forward_np` compiles one forward pass per network structure and reads the network's current parameters at each call, so optimizer steps need no recompilation. `_pad` and `_bucket` pad a batch to the next power of two (at least 64 rows), so a compiled update is reused across batches of different lengths; `padded` pads several arrays of one batch together and adds a mask that marks the real rows, so that an update can leave the padding out of its means. `shared_adam` keeps one `optax.adam` per learning rate, so agents created with the same learning rate share their compiled updates, and `plain_sgd` does the same for plain SGD, which some experiments use in place of Adam.

In [ ]:

_forwards = {}                          # one compiled forward per net structure


import weakref
_splits = weakref.WeakKeyDictionary()  # per net: its forward and variables

def forward_np(net, x):
    """net(x) for numpy x, as numpy, through one compiled forward per net
    structure and input shape. The net is split once and its variables are
    kept, so each call reads their current values: optimizer steps show up
    without splitting again."""
    if net not in _splits:
        graph, state = nnx.split(net)
        tree = jax.tree.structure(state)
        if (graph, tree) not in _forwards:
            _forwards[graph, tree] = jax.jit(lambda leaves, x: nnx.merge(
                graph, jax.tree.unflatten(tree, leaves))(x))
        _splits[net] = _forwards[graph, tree], jax.tree.leaves(
            state, is_leaf=lambda v: isinstance(v, nnx.Variable))
    forward, variables = _splits[net]
    leaves = [v.get_raw_value() for v in variables]
    return jax.tree.map(np.asarray, forward(leaves, x))

def _pad(x, size):
    """Zero-pad axis 0 of an array to `size` rows."""
    x = np.asarray(x)
    return np.pad(x, ((0, size - len(x)),) + ((0, 0),) * (x.ndim - 1))

def _bucket(n):
    """Padded batch length: the next power of two, at least 64, so that a
    compiled pass is reused across batches of different lengths."""
    return 1 << max(6, (n - 1).bit_length())

def padded(*arrays):
    """The arrays zero-padded along axis 0 to a shared bucket length, plus a
    0/1 mask of the real rows, so that one compiled pass serves every batch
    length."""
    n = len(arrays[0])
    size = _bucket(n)
    mask = (np.arange(size) < n).astype(np.float32)
    return (*(_pad(x, size) for x in arrays), mask)


_adams = {}

def shared_adam(lr):
    """One optax.adam per learning rate, shared by all agents: compiled
    updates are keyed on the optimizer, so a new agent reuses the compiled
    code instead of compiling it again."""
    return _adams.setdefault(lr, optax.adam(lr))


_sgds = {}

def plain_sgd(net, lr):
    """A plain SGD optimizer over `net`, one optax.sgd per learning rate."""
    return nnx.Optimizer(net, _sgds.setdefault(lr, optax.sgd(lr)),
                         wrt=nnx.Param)

## Feature Networks for Image Scores

The generative chapters score images in the features of a small CIFAR-10 classifier (that section). `load_cifar10_arrays` returns the CIFAR-10 training and test images, scaled to $[-1, 1]$, with their labels. `FeatureCNN` is a classifier of three convolution blocks, each with batch normalization, ReLU and $2 \times 2$ max pooling, followed by global average pooling and a linear head; its `features` method returns the pooled features, and its `resolution` argument sets the size to which it first resizes its inputs, low-pass filtering them when it shrinks them. `train_feature_cnn` trains it with Adam under a one-cycle learning-rate schedule on random horizontal flips and, optionally, random shifts.

In [ ]:

def load_cifar10_arrays():
    """CIFAR-10 on the default device, pixel values scaled to [-1, 1].
    Returns train_X, train_y, test_X, test_y."""
    import pickle, warnings  # not in the d2l library preamble
    d2l.DATA_HUB['cifar10'] = (
        'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz',
        '874905e36347c8536514d0a26261acf3bff89bc7')
    cifar_dir = d2l.download_extract('cifar10', 'cifar-10-batches-py')
    def load_batch(name):
        with open(f'{cifar_dir}/{name}', 'rb') as f, warnings.catch_warnings():
            warnings.simplefilter('ignore')   # legacy NumPy pickle format
            batch = pickle.load(f, encoding='bytes')
        X = batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        return (jnp.asarray(X, dtype=jnp.float32) / 127.5 - 1,
                jnp.asarray(batch[b'labels']))
    Xs, ys = zip(*[load_batch(f'data_batch_{i}') for i in range(1, 6)])
    test_X, test_y = load_batch('test_batch')
    return jnp.concatenate(Xs), jnp.concatenate(ys), test_X, test_y


class FeatureCNN(nnx.Module):
    """Three conv blocks, global average pooling, and a linear head. Inputs
    are first resized bilinearly to resolution x resolution."""
    def __init__(self, feature_dim=128, resolution=32, rngs=None):
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.resolution = resolution
        widths = [3, 32, 64, feature_dim]
        # Weights and biases from U(+-1/sqrt(fan_in)); with Flax's default
        # initialization the same images score FDs about 15% lower
        self.convs = nnx.List(
            [d2l.Conv(c_in, c_out, (3, 3), padding='SAME', rngs=rngs)
             for c_in, c_out in zip(widths[:-1], widths[1:])])
        # Running averages decay 0.9; the variance is computed in two passes
        self.norms = nnx.List([nnx.BatchNorm(c, momentum=0.9,
                                             use_fast_variance=False,
                                             rngs=rngs)
                               for c in widths[1:]])
        self.head = d2l.Linear(feature_dim, 10, rngs=rngs)

    def features(self, x):
        if x.shape[1] != self.resolution:
            x = jax.image.resize(
                x, (x.shape[0], self.resolution, self.resolution, x.shape[3]),
                method='bilinear')
        for conv, norm in zip(self.convs, self.norms):
            x = nnx.max_pool(nnx.relu(norm(conv(x))),
                             window_shape=(2, 2), strides=(2, 2))
        return x.mean(axis=(1, 2))

    def __call__(self, x):
        return self.head(self.features(x))


def train_feature_cnn(cnn, X, y, num_epochs=15, max_lr=0.003,
                      batch_size=256, shift=0, seed=0):
    """Train on random horizontal flips, plus random shifts of up to `shift`
    pixels, with a one-cycle learning rate; end in evaluation mode."""
    schedule = optax.cosine_onecycle_schedule(
        num_epochs * (len(X) // batch_size), max_lr)
    opt = nnx.Optimizer(cnn, optax.adam(schedule), wrt=nnx.Param)
    h, w = X.shape[1:3]

    @nnx.jit
    def step(cnn, opt, Xb, yb, key):
        k1, k2 = jax.random.split(key)
        flip = jax.random.bernoulli(k1, 0.5, (len(Xb), 1, 1, 1))
        Xb = jnp.where(flip, Xb[:, :, ::-1], Xb)
        a, b = jax.random.randint(k2, (2,), 0, 2 * shift + 1)
        Xb = jnp.pad(Xb, ((0, 0), (shift, shift), (shift, shift), (0, 0)),
                     mode='reflect')
        Xb = jax.lax.dynamic_slice(Xb, (0, a, b, 0), (len(Xb), h, w, 3))
        def loss_fn(cnn):
            return optax.softmax_cross_entropy_with_integer_labels(
                cnn(Xb), yb).mean()
        opt.update(cnn, nnx.grad(loss_fn)(cnn))

    cnn.train()
    key = jax.random.key(seed)
    for epoch in range(num_epochs):
        key, kp = jax.random.split(key)
        order = jax.random.permutation(kp, len(X))
        for i in range(0, len(order) - batch_size + 1, batch_size):
            key, ka = jax.random.split(key)
            idx = order[i:i + batch_size]
            step(cnn, opt, X[idx], y[idx], ka)
    cnn.eval()

## Training Utilities

The following legacy helper functions are retained for backward compatibility:

In [5]:

def linreg(X, w, b):
    """The linear regression model."""
    return d2l.matmul(X, w) + b

def squared_loss(y_hat, y):
    """Squared loss."""
    return (y_hat - d2l.reshape(y, y_hat.shape)) ** 2 / 2

class ArrayDataLoader(d2l.ArrayLoader):
    """`d2l.ArrayLoader` that yields JAX arrays and also accepts a dataset.

    Pass the arrays as positional arguments, or one dataset object with
    `__len__` and `__getitem__` whose items are tuples of fields. Like
    `d2l.ArrayLoader`, it keeps the last, partial minibatch unless
    `drop_last=True`."""
    def __init__(self, *args, batch_size=32, shuffle=False, drop_last=False):
        if len(args) == 1 and not hasattr(args[0], 'shape'):  # A dataset
            items = [args[0][i] for i in range(len(args[0]))]
            if isinstance(items[0], (tuple, list)):
                args = tuple(np.array([item[j] for item in items])
                             for j in range(len(items[0])))
            else:
                args = (np.array(items),)
        super().__init__(args, batch_size, shuffle=shuffle,
                         drop_last=drop_last)

    def __iter__(self):
        for batch in super().__iter__():
            yield tuple(jnp.asarray(a) for a in batch)

def load_array(data_arrays, batch_size, is_train=True):
    """Minibatches of JAX arrays, reshuffled each pass if is_train."""
    return d2l.ArrayDataLoader(*data_arrays, batch_size=batch_size,
                               shuffle=is_train)

class Animator:
    """For plotting data in animation."""
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        if legend is None:
            legend = []
        d2l.use_svg_display()
        self.fig, self.axes = d2l.plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        self.config_axes = lambda: d2l.set_axes(
            self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)

class Accumulator:
    """For accumulating sums over `n` variables."""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def tokenize(lines, token='word'):
    """Split text lines into word or character tokens."""
    assert token in ('word', 'char'), 'Unknown token type: ' + token
    return [line.split() if token == 'word' else list(line) for line in lines]

def truncate_pad(line, num_steps, padding_token):
    """Truncate or pad sequences."""
    if len(line) > num_steps:
        return line[:num_steps]
    return line + [padding_token] * (num_steps - len(line))

def download_extract(name, folder=None):
    """Download and extract a zip/tar file."""
    fname = download(name)
    base_dir = os.path.dirname(fname)
    data_dir, ext = os.path.splitext(fname)
    target = os.path.join(base_dir, folder) if folder else data_dir
    marker = fname + '.extracted'
    if os.path.exists(marker):
        return target
    if ext == '.zip':
        fp = zipfile.ZipFile(fname, 'r')
    elif ext in ('.tar', '.gz'):
        fp = tarfile.open(fname, 'r')
    else:
        assert False, 'Only zip/tar files can be extracted.'
    fp.extractall(base_dir)
    open(marker, 'w').close()
    return target

def evaluate_loss(net, data_iter, loss):
    """Evaluate the loss of a model on the given dataset."""
    metric = d2l.Accumulator(2)
    for X, y in data_iter:
        out = net(X)
        y = d2l.reshape(y, out.shape)
        l = loss(out, y)
        metric.add(d2l.reduce_sum(l), d2l.size(l))
    return metric[0] / metric[1]

In [6]:
def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5,
                cmap=None):
    """Plot a list of images (cmap colors single-channel images)."""
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        try:
            img = d2l.numpy(img)
        except:
            pass
        ax.imshow(img, cmap=cmap)
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i])
    return axes

`show_heatmaps` draws matrices as grids of colored cells, one panel per
matrix. Its argument has the shape (rows of panels, columns of panels,
height, width). By default all panels share one color scale and one
colorbar; `cbar='each'` gives every panel its own scale and colorbar, and
`cbar=False` draws none. Ticks mark integer rows and columns, and
`xticklabels` and `yticklabels` name them, e.g., by token. It displays the
confusion matrix of that section and the attention weights
of that section.

In [ ]:
def show_heatmaps(matrices, xlabel, ylabel, titles=None,
                  figsize=(2.5, 2.5), cmap='Reds', cbar=True,
                  xticklabels=None, yticklabels=None):
    """Show heatmaps of matrices; cbar is True (shared), 'each' or False."""
    d2l.use_svg_display()
    matrices = d2l.numpy(matrices)
    num_rows, num_cols, height, width = matrices.shape
    fig, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize,
                                 sharex=True, sharey=True, squeeze=False)
    # All panels share one color scale (one norm) unless each gets its own
    lims = {} if cbar == 'each' else dict(norm=d2l.plt.Normalize(
        matrices.min(), matrices.max()))
    for i, (row_axes, row_matrices) in enumerate(zip(axes, matrices)):
        for j, (ax, matrix) in enumerate(zip(row_axes, row_matrices)):
            pcm = ax.imshow(matrix, cmap=cmap, **lims)
            if cbar == 'each':
                fig.colorbar(pcm, ax=ax, shrink=0.6)
            if i == num_rows - 1:
                ax.set_xlabel(xlabel)
            if j == 0:
                ax.set_ylabel(ylabel)
            if titles:
                ax.set_title(titles[j])
    # Shared axes: setting the ticks of one panel sets them for all
    for axis, n, labels in ((ax.xaxis, width, xticklabels),
                            (ax.yaxis, height, yticklabels)):
        if labels is None:
            axis.set_major_locator(d2l.plt.MaxNLocator(
                'auto', integer=True, steps=[1, 2, 2.5, 5, 10]))
        else:
            axis.set_ticks(range(n), labels)
    if cbar is True:
        fig.colorbar(pcm, ax=axes, shrink=0.6);

In [8]:

import os
import requests
import zipfile
import tarfile
import hashlib

def download(url, folder='../data', sha1_hash=None):
    """Download a file to folder and return the local filepath."""
    if not url.startswith('http'):
        # For back compatibility
        url, sha1_hash = DATA_HUB[url]
    os.makedirs(folder, exist_ok=True)
    fname = os.path.join(folder, url.split('/')[-1])
    # Check if hit cache
    if os.path.exists(fname) and sha1_hash:
        sha1 = hashlib.sha1()
        with open(fname, 'rb') as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname
    # Download
    import tempfile
    print(f'Downloading {fname} from {url}...')
    r = requests.get(url, stream=True, verify=True)
    fd, tmp = tempfile.mkstemp(dir=folder)
    try:
        with os.fdopen(fd, 'wb') as f:
            f.write(r.content)
        os.replace(tmp, fname)
    except BaseException:
        os.unlink(tmp)
        raise
    return fname

def extract(filename, folder=None):
    """Extract a zip/tar file into folder."""
    import tempfile, shutil
    base_dir = os.path.dirname(filename)
    _, ext = os.path.splitext(filename)
    assert ext in ('.zip', '.tar', '.gz'), 'Only support zip/tar files.'
    opener = zipfile.ZipFile if ext == '.zip' else tarfile.open
    if folder is None:
        folder = base_dir
    tmp = tempfile.mkdtemp(dir=folder)
    try:
        with opener(filename, 'r') as fp:
            fp.extractall(tmp)
        for name in os.listdir(tmp):
            src = os.path.join(tmp, name)
            dst = os.path.join(folder, name)
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            os.rename(src, dst)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

## Sequence and Attention Utilities

### Embedding Gradients for Small Vocabularies

The gradient of an embedding lookup adds the gradient at each position to
the row of that position's token, and XLA performs these additions with
atomic operations. In a minibatch of text, thousands of positions hold the
same few frequent tokens, such as the bytes of a space and of `e`. Their
additions to the same row wait for one another, and on a GPU the sum can
take tens of times as long as for random tokens. For vocabularies of up to
1,024 tokens, `Embed` computes the same sum as one matrix product of the
one-hot rows of the tokens with the gradients. It is otherwise an
`nnx.Embed`, and the JAX models of the book use it for their token tables.

In [ ]:
@jax.custom_vjp
def embedding_rows(table, X):
    """The rows X of table."""
    return table[X]

def _embedding_rows_fwd(table, X):
    return table[X], (table, X)

def _embedding_rows_bwd(residuals, grad):
    """Gradient of the table: the one-hot rows of X times the gradients."""
    table, X = residuals
    one_hot = jax.nn.one_hot(X, len(table), dtype=grad.dtype)
    return jnp.einsum('...v,...d->vd', one_hot, grad,
                      precision='highest').astype(table.dtype), None


embedding_rows.defvjp(_embedding_rows_fwd, _embedding_rows_bwd)

class Embed(nnx.Embed):
    """nnx.Embed whose gradient is a matrix product for small vocabularies."""
    def __call__(self, X):
        table = self.embedding[...]
        if len(table) > 1024:  # A one-hot row would cost too much
            return super().__call__(X)
        return embedding_rows(table, X).astype(self.dtype or table.dtype)

## A Note on Framework Coverage

The legacy helpers in this section (`evaluate_accuracy`, `train_ch6`,
`train_seq2seq`, `predict_seq2seq`, `MaskedSoftmaxCELoss`) are kept for
parity with the original D2L implementation, which predates the unified
`d2l.Trainer` class introduced in this edition. They are not provided for
JAX, and the PyTorch module includes only the subset used outside the
`Trainer` workflow. If you are reading the JAX tab,
the corresponding chapters use `d2l.Trainer.fit(model, data)` end-to-end;
the corresponding per-batch logic is implemented in
`Trainer.fit_epoch` and the `@d2l.add_to_class` extensions defined
alongside it. The MXNet and TensorFlow tabs additionally retain
`evaluate_accuracy` because some of their earlier-chapter snippets call
it directly; for PyTorch and JAX,
use `Trainer.validation_step` (or its return value) instead.

[Discussions](https://d2l.discourse.group/t/36849)